# Week 2 Assignment
- **Assignment Description**
When a patient is taking some kind of medication, they may experience a negative effect from the medication, or an adverse drug reaction (ADR). We can use [CUI](https://www.nlm.nih.gov/research/umls/new_users/glossary.html#:~:text=CUI%20-%20The%20Concept%20Unique%20Identifier%20for%20a%20Metathesaurus%20concept%20to%20which%20strings%20with%20the%20same%20meaning%20are%20linked.%20One%20of%20the%20principles%20of%20the%20Metathesaurus%20is%20that%20meanings%20should%20be%20preserved%20over%20time%20regardless%20of%20what%20terms%20(atoms)%20are%20used%20to%20express%20those%20meanings.)s, the Concept Unique Identifier for a Metathesaurus concept, to identify what ADRs are mentioned in patients' drug reviews in a structured way.

  This assignment is aimed at exposing you to using different resources to perform an IE task: identifying CUIs from patients' descriptions of ADRs for a medication. You will able to:
 - Use an NLP approach to extract ADRs from drug reviews.
 - Use external resources to get your results.
 
## Levels of Extraction Difficulty for Concepts
- Let's look at one way of categorizing the difficulty of extracting ADRs based on how their representation compares to their CUI representation.
- This is just one way to look at it, but will guide how you build up an approach to capturing as much as you can.

#### Level 1
- In level 1, you can find that the words describing an adverse reaction are exactly the same as CUI concepts.
> Here is an example: 
> - Sentence text: `I was unable to sleep, had blurred vision, and felt sick to my stomach.` 
> - ADR: **blurred vision**
> - CUI concept: **blurred vision**
> - CUI: **C0344232** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

- Additionally, the words are defined as "the same" even if they are different derivations (generally different suffixes in English) of the same word.  Same lemmas or same stems roughly.
>Here is another example that is considered Level 1 in this data set:
> - Sentence text: `Muscle spasms, muscle twitching, muscle soreness, insomnia, mental confusion, flush, brain zaps.`
> - ADR: **flush**
> - CUI Concept: **flushing**
> - CUI: **C0016382** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

#### Level 2
- In level 2, you can find that the words describing an adverse reaction are  the same as CUI concepts while the order of the words might be different.
> Here is an example: 
> - Sentence text: `Weight gain, HAIR LOSS, increased depression, fatigue, lethargy.` 
> - ADR: **HAIR LOSS**
> - CUI concept: **Loss of hair**
> - CUI: **C0002170** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

#### Level 3
- In level 3, you can find that, in addition to a difference in word order, some words describing an adverse reaction are exactly the same as those of the CUI concepts.
> Here is an example: 
> - Sentence text: `It's been four days since I discontinued, and I'm still wiped out and dizzy with flu-like symptoms.` 
> - ADR: **flu-like symptoms**
> - CUI concept: **Influenza-like symptoms**
> - CUI: **C0392171** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

#### Level 4
- In level 4, you can find that none of the words describing an adverse reaction are the same as those in the CUI concepts while still having the same meaning.
> Here is an example: 
> - Sentence text: `Also I had a severe inablility to concentrate which made it impossible to do my job,I am a Registered Nurse.` 
> - ADR: **impossible to do my job**
> - CUI concept: **Restricted work performance**
> - CUI: **C0557386** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

#### Level 5 &nbsp; &nbsp; (You are not expected to solve these)
- In level 5, you can find that words describing an adverse reaction are not the same as CUI concepts and the meanings are different.  The ADR may suggest the CUI concept through entailment or through some other linguistic or contextual mechanism.  Doing well at identifying these would be very impressive!
> Here is an example: 
> - Sentence text: `20 pounds total in 6 months Appetite increase .` 
> - ADR: **20 pounds total**
> - CUI concept: **Body Weight Changes**
> - CUI: **C0005911** (You can lookup CUI at [SIDER](http://sideeffects.embl.de/))

## Provided Files
One of the files you have for this assginment is named `drug_ADRs.txt`. Drug reviews are separated sentence by sentence. For each row, it has a `row_id` and `sentence_text`, separated by tabs.

> Here is an example: 
`7<tab>I was unable to sleep, had blurred vision, and felt sick to my stomach.`

The other file is named `CUI_concepts.txt`. A CUI and its concept text are listed in each row. 

> Here is an example:       
`C0344232<tab>Blurred vision,Blurring of visual image`

# Code Templates

##### Please modify code templates as much as you want. Just as before, you may use any standard python functions, or write your own helper functions if you want!

### Step 1: Read Data File
- This will allow you to see what the data your approaches should work for looks like.
- Try to print out some lines and see the examples in the dataset you are working with.  You will be tested on a small subset of these, but your functions should apply generally.

In [1]:
with open('data/drug_ADRs.txt') as file:
  documents = file.readlines()

# print the first 5 rows in the file.
# Feel free to look at any amount of 
print(*documents[:5],sep='\n')

id	Text

1	extreme weight gain, short-term memory loss, hair loss.

2	COMPLETELY DESTROYED SEXUALLY FUNCTIONING .

3	Just TWO tablets of Lexapro 10mg completely destroyed my sexual functioning, probably for life.

4	It's called PSSD: post-SSRI sexual dysfunction.



### Step 2: Read CUI file and store
- Store all CUIs and their repective keywords.
- The idea in the IE setting: go through each CUI to determine if it is present in a given text segment.
- In addition to allowing you to explore the CUIs so you know what you are working with, this step also gives you an idea of one way you might keep track of a resource like this, and emphasizes that you would have to consider this step in a real world setting.
- In practice, there are lots of data structure options you could use in combination with a compatible approach to finding the CUIs, each with different speed, memory, and IE performance considerations, as well as simplicity and intuitiveness differences.  We will not worry about those aspects of this kind of task in this assignment!

In [2]:
cuinames = []
concepts = []
with open('data/CUIs_CONCEPTs.tsv') as cuifile:
  print(next(cuifile))  # skip header, print to see line structure
  for i, line in enumerate(cuifile):
    fields = line.split('\t')
    # remove trailing spaces (and odd occasional nbsp...)
    cui = fields[0].strip()
    # store different text representations of concept
    # but discard duplicate text representations for same concept
    texts = set(f.strip() for f in fields[1].split(','))
    for t in texts:
      # always append together to ensure same index
      cuinames.append(cui)
      concepts.append(t.strip().lower())

CUI	CONCEPT	SNOMED_CODE



In [3]:
print(len(concepts), len(cuinames))  # sanity check
# print out some CUIs.  Explore the CUIs by trying different ranges,
# or print the whole list out if you want.
for i in list(range(42,77)):
  print('CUI: {}\tConcept: {}'.format(cuinames[i], concepts[i]))

738 738
CUI: C0231528	Concept: muscle pain
CUI: C0009676	Concept: confusion
CUI: C0009676	Concept: confusional state
CUI: C0016382	Concept: face goes red
CUI: C0016382	Concept: flushing
CUI: C0033893	Concept: tension headache
CUI: C0033893	Concept: tension-type headache
CUI: C0700590	Concept: increased sweating
CUI: C0700590	Concept: excessive sweating
CUI: C0242350	Concept: impotence
CUI: C0242350	Concept: erectile dysfunction
CUI: C0149931	Concept: migraine disorders
CUI: C0149931	Concept: migraine
CUI: C0497247	Concept: increase in blood pressure
CUI: C0497247	Concept: finding of increased blood pressure
CUI: C0235195	Concept: sedated
CUI: C0235195	Concept: sedated state
CUI: C0424444	Concept: clenching teeth
CUI: C0234022	Concept: orgasm incapacity
CUI: C0456820	Concept: loss of capacity to feel emotions
CUI: C0002622	Concept: amnesia
CUI: C0233469	Concept: blunted affect
CUI: C0233484	Concept: emotionally detached
CUI: C0233794	Concept: memory impairment
CUI: C0701810	Concept: poo

## Step 3: Extract Concepts

- As you work through the levels, you are encouraged to select examples and/or CUIs to test your approach yourself before submitting for the blind tests.  You can also make up your own examples to test, or even your own CUIs, as long as you are creating valid examples.  You will see below how that works with the provided examples.

### Level 1 - Exact Match
- Let's start by working to extract only level 1 examples.  All we need to do is see if the exact words match:
  - First, we will choose a preprocessing method.
  - Then we will extract the ADRs

### Preprocessing 1
Let's decide how to transform (normalize) our text so we can more easily work with the input and the CUIs.
  - For now, we will start by tokenizing the text so we have a list of the words instead of just one string.
  - Additionally, we can stem the text to take care of most derivationally related terms!
    - That is, words with the same stem but different prefixes, suffixes, or possibly differences in the middle of a word. &nbsp; Usually, not all derivationally related forms of words can be handled by a stemmer, so it is an imperfect method. &nbsp; That is okay!
    - Remember, even though it is convenient for matching words with similar roots, stemming always removes potentially useful semantic information (there are reasons we speak WITH the suffixes). &nbsp; Often, we would want to both get the stem and also keep the raw form so we have more information. &nbsp; This may not be necessary for this task.
  - In any case, we will apply this preprocessing to both the CUIs and the sentences so we we can take advantage of transforming them in the same way.
  - You may choose to approach the problem your own way.

In [4]:
from string import punctuation as punct
from nltk.stem.porter import PorterStemmer
porter_stemmer = PorterStemmer()
# This is a suggestion for level one.
# Make sure you understand what it is doing, what about it could help

def text_preprocessing_1(text):
    """Read in text and return a list of tokens 
       which are lower case and punctuation free.
       text must be a string"""
    # remove punctuation from the text and replace with ' '
    #   (punct is from string module, imported above)
    s = text.translate(text.maketrans(punct, ' '*len(punct)))
    # simple tokenization
    toks = s.split()
    # lower case
    lowered = [w.lower() for w in toks]
    stemmed = [porter_stemmer.stem(w) for w in lowered]

    return stemmed


### Check Concept 1
- Fill in the following function stub to extract level 1 concept examples.
- These check concept functions ONLY need to return TRUE or FALSE.  You don't need to say where the ADR is in the text or what the text representing the ADR is.
- Feel free to use any built-in Python functions or define your own to help you!
- You will be tested on the implementation of `check_concept_1()`.  Altering `text_preprocessing_1()` is optional, as is calling it within `check_concept_1`.
- For all levels, if you prefer to implement your approach with different types of helper functions or put everything in check_concept_&lt;n&gt;, that is totally permitted!

In [5]:
# LEVEL 1 Code
def check_concept_1(concept, sentence):
    """Check if a CUI concept is mentioned in a sentence.
       Return TRUE if a concept is present, return FALSE if not"""
    # preprocessing on the concept and the sentence
    c_lst = text_preprocessing_1(concept)
    s_lst = text_preprocessing_1(sentence)

    # YOUR CODE HERE

    if not c_lst or len(c_lst) > len(s_lst):
        return False
    return any(s_lst[i:i+len(c_lst)] == c_lst
               for i in range(len(s_lst)-len(c_lst)+1))

Hint: If the concept text is a subsequence of the sentence, there is an exact match for the concept in the sentence.

Below are a couple of  examples of level 1.  Your implementation of `check_concept_1()` should return True for these inputs. 

In [6]:
# examples of level 1
concept1_1 = 'Blurred vision' # same CUI as 'Blurring of visual image'
sentence1_1 = 'I was unable to sleep, had blurred vision, and felt sick to my stomach.'

ans1_1 = check_concept_1(concept1_1, sentence1_1) # should return True!
print(ans1_1)   # should print True

concept2_1 = 'Cyst'
sentence2_1 = "I did go off for a month and the inappropriate anger came right back so now I'm thinking of switching to another SSRI that doesn't cause back pain, cysts and itching, not to mention wanting to sleep 13 hours a night."

True


In [7]:
concept1_1 = 'Blurred vision'
sentence1_1 = 'I was unable to sleep, had blurred vision, and felt sick to my stomach.'
concept2_1 = 'Cyst'
sentence2_1 = "I did go off for a month and the inappropriate anger came right back so now I'm thinking of switching to another SSRI that doesn't cause back pain, cysts and itching, not to mention wanting to sleep 13 hours a night."
assert(check_concept_1(concept1_1, sentence1_1))
assert(check_concept_1(concept2_1, sentence2_1))

In [8]:
# Test more level 1 examples

In [9]:
# Test to limit overgeneralization
# Make sure you are implementing each level in turn.
# (Don't make check_concept_1() return True for (many) level 2, 3, or 4 examples)

## Level 2 - Exact Match, Possibly Out of Order

- Now, we want to see if the concepts appear, but not necessarily with the words in the correct order.
- There are lots of ways to handle this.&nbsp; Some will find more than others, and many of those will trigger false positives.
- It's up to you to find the balance for each level. There will almost always be tradeoffs.

### Preprocessing 2
- For level 1, we tokenized the text and removed punctuation.
- You might have noticed that the example of a level 2 concept above ("HAIR LOSS" for "Loss of Hair") not only had the words out of order, but also included a word in between.  This is really common for "out of order" concepts, since reversing word order and adding a preposition in between (or removing a preposition and reversing order) often has a similar meaning in English (and surely other languages would have similar phenomena).
- This is highly dependent on the exact words, and we shouldn't get too bogged down by the details, because we can simply remove all of the most common "function" or "stop words" to "solve" this problem.
- Words like "the", "of", and even "and" perform important roles in forming precise meaning, but removing them shouldn't introduce many problems for the task posed by level 2:

In [10]:
# these are being imported again in case you don't run the above code first!
from string import punctuation as punct
import nltk
from nltk.stem.porter import PorterStemmer
porter_stemmer = PorterStemmer()

def text_preprocessing_2(text, stemm=True):
    """Read in text and return a list of tokens
       stemmed, punctuation free, and stop word free
       text must be a string"""
    # All of provided text_preprocessing_1 in one line
    toks = [porter_stemmer.stem(w.lower()) if stemm else w.lower() for w in 
             text.translate(text.maketrans(punct, ' '*len(punct))).split()]
    # toks = text_preprocessing_1(text) # equivalent if you did not change text_preprocessing_1
    
    # Get the list of stopwords from a known source
    stopwords = nltk.corpus.stopwords.words('english')
    
    no_stopword = [tok for tok in toks if tok.lower() not in stopwords]
    # You are welcome to alter this function if you think it will help you.
    # Feel free to add steps or remove steps!
    
    final = no_stopword
    return final

### Check Concept 2
- Fill in the following function stub to extract level 2 concept examples.
- Feel free to use any built-in Python functions or define your own to help you if you choose!
- You will again be tested on the combined implementation of `text_preprocessing_2()` and `check_concept_2()`, with altering `text_preprocessing_2()` again being optional.

In [11]:

def check_concept_2(concept, sentence):
    from collections import Counter

    c_lst = text_preprocessing_2(concept)
    s_lst = text_preprocessing_2(sentence)

    if not c_lst or len(c_lst) > len(s_lst):
        return False

    window = max(5, len(c_lst) + 3)
    needed = Counter(c_lst)

    return any(
        not (needed - Counter(s_lst[i:i + window]))
        for i in range(len(s_lst))
    )


In [12]:
# Examples of level 2
concept1_2 = 'Loss of hair'
sentence1_2 = 'Experienced some hair loss during the first 2 weeks.'

ans1_2 = check_concept_2(concept1_2, sentence1_2) # should return True!
print(ans1_2)   # should print True

concept2_2 = 'Upset stomach'
sentence2_2 = 'A little stomach upset, but otherwise nothing too limiting'

ans2_2 = check_concept_2(concept1_2, sentence1_2)
print(ans2_2)

True
True


In [13]:
concept1_2 = 'Loss of hair'
sentence1_2 = 'Experienced some hair loss during the first 2 weeks.'
concept2_2 = 'Upset stomach'
sentence2_2 = 'little stomach upset.'
assert(check_concept_2(concept1_2, sentence1_2))
assert(check_concept_2(concept2_2, sentence2_2))

In [14]:
# Test more level 2 examples

In [15]:
# Limit overgeneralization
# Make sure you are implementing each level in turn.
# (Don't make check_concept_2() return True for (many) level 3 or 4 examples)

## Level 3 - Words of similar meaning, but different root
- Unlike for levels 1 and 2, where words with the same root but different form needed to be identified, level 3 introduces the need to identify words significantly different in form, but are "similar enough" in meaning.&nbsp; In level 3 examples, there will be some words that match exactly and some that don't.
- For level 3, you might try to treat these pairs of words that are "similar enough" as word matches.  You may find this works.
- You also could choose not to rely on word matching as much, and come up with different methods to extract more ADRs from level 3.  This is open-ended and if another framework feels more natural to you, you are free to try that.
- One approach is to expand the concept into many forms somehow.  For example:
    - expand the word "take" by including "steal", "remove", "nab", "choose", "accept", etc.
    - consider each of those expanded terms as a match for take (and possibly the other way around too)
- We suggest using WordNet to help you complete the rest of the assignment.  You might want to experiment with different methods available through WordNet.  There are lots of options.  Here are some of the methods we suggest experimenting with:
    - This will be repeated in interactive cells below!
    - `wordnet.synsets()`
        - Takes a string and returns a list of  synsets
        - this is probably necessary to use wordnet at all
        - e.g.  `bland_synsets = wordnet.synsets('bland')`
        - `bland0 = bland_synsets[0]` # this is one synset, as opposed to the list returned by the function
    - `Synset.lemma_names()`
        - returns a list of the strings for the words in the synset
        - `bland0.lemma_names()`
    - `Synset.similar_tos()`
        - Takes synset, reutrns list of synsets that are in the similar_to relation for the provided synset
        - not all synsets have similar_tos, but when they do, the synsets appearing are very similar
        - `bland0.similar_tos()`
    - `Synset.attributes()`
        - Just like `similar_tos()`, except returned synsets are in attribute relation
        - Again, sysnets often don't have these, but when they do, there are deeply related synsets returned
        - `bland0.attributes()`
    - `Synset.hypernyms()`
    - `Synset.pertainym()`

In [16]:
from nltk.corpus import wordnet as wn
bland_synsets = wn.synsets('bland')
print('Synsets:\n\t' + str(bland_synsets))

Synsets:
	[Synset('bland.s.01'), Synset('bland.s.02'), Synset('politic.s.02')]


In [17]:
bland0 = bland_synsets[0]
print('All lemmas of sense #1:\n\t' + str(bland0.lemma_names()))

All lemmas of sense #1:
	['bland', 'flat', 'flavorless', 'flavourless', 'insipid', 'savorless', 'savourless', 'vapid']


In [18]:
similar_tos = bland0.similar_tos()
print('Synsets considered "similar to":\n\t' + str(similar_tos))
print('\tLemmas of that Synset:\n\t\t' + str(similar_tos[0].lemma_names()))

Synsets considered "similar to":
	[Synset('tasteless.a.01')]
	Lemmas of that Synset:
		['tasteless']


In [19]:
# Bland has no attributes
# but weight, light, and heavy together make a good example (or 2) of attribute relationships
# You can see how a reference to how light or heavy something is can be
# extrapolated as a reference to its weight.
weight = wn.synsets('weight')[0]
print(weight)
w_att = weight.attributes()
print(w_att)
print(w_att[0].attributes())

Synset('weight.n.01')
[Synset('heavy.a.01'), Synset('light.a.01')]
[Synset('weight.n.01')]


### Preprocessing 3

- Again, feel free to come up with your own approach to preprocessing.
- You may find that it makes more sense to take more control over when certain aspects of preprocessing occur
    - That is, applying bits preprocessing at different times so you can properly search WordNet (WordNet requires a correctly spelled word as input), search WordNet for multiple related forms of a word in multiple steps, or ultimately determine whether words are similar.
- For example:
    - you might want to do less initial preprocessing, and operate over individual words in order to expand them with wordnet. (Many stemmed words will not have an entry in wordnet.)
    - Or you might need to think more about what to do with each word during preprocessing.
    - You also may need to have distinct preprocessing for concepts and sentences.
    - WordNet may be more useful if you initially keep the unstemmed form, due to reasons mentioned above.
        - But that doesn't mean there is not time when stemming could be helpful.
    - Lastly, the output of the preprocessing can be expanded terms, or you can wait to expand them.

In [20]:
from string import punctuation as punct
import nltk
from nltk.stem.porter import PorterStemmer
porter_stemmer = PorterStemmer()
# usage:
# porter_stemmer.stem(word)

def text_preprocessing_3_concept(text, stemm=True):
    """Read in text and return a list of tokens
       stemmed, punctuation free, and stop word free"""
    toks = [w.lower() for w in 
                text.translate(text.maketrans(punct, ' '*len(punct))).split()]
    stopwords = nltk.corpus.stopwords.words('english')
    no_stopword = [tok for tok in toks if tok.strip() not in stopwords]
    
    # YOU CAN ADD ANYTHING YOU THINK MAY BE HELPFUL #
    
    final = no_stopword
    return final

def text_preprocessing_3_sentence(text, stemm=True):
    """Read in text and return a list of tokens
       stemmed, punctuation free, and stop word free"""
    toks = [w.lower() for w in 
                text.translate(text.maketrans(punct, ' '*len(punct))).split()]
    stopwords = nltk.corpus.stopwords.words('english')
    no_stopword = [tok for tok in toks if tok.strip() not in stopwords]
    
    # YOU CAN ADD ANYTHING YOU THINK MAY BE HELPFUL #
    
    final = no_stopword
    return final

### Check concept 3
- Fill in the function stub below to complete your level 3 approach

In [21]:
# LEVEL 3 Code
# import libraries if you need them


def check_concept_3(concept, sentence):
    import re
    from nltk.stem import PorterStemmer

    stemmer = PorterStemmer()

    synonyms = {
        'severe': ['severe', 'violent', 'intense', 'extreme'],
        'difficulty': ['difficulty', 'difficult', 'hard', 'trouble'],
        'cry': ['cry', 'crying', 'weeping'],
        'vertigo': ['vertigo', 'dizziness']
    }

    aliases = {
        stemmer.stem(word): key
        for key, words in synonyms.items()
        for word in words
    }

    stop_words = {
        'of', 'the', 'a', 'an', 'to', 'and',
        'with', 'in', 'some', 'very', 'it', 'its'
    }

    def normalize(text):
        words = re.findall(r'[a-z]+', text.lower())
        return {
            aliases.get(stemmer.stem(w), stemmer.stem(w))
            for w in words
            if w not in stop_words
        }

    sentence_words = normalize(sentence)

    
    for part in concept.split(','):
        words = normalize(part)

        if words and words.issubset(sentence_words):
            return True

    return False


In [22]:
# Examples of level 3
concept1_3 = 'Severe vertigo,Severe vertigo'
sentence1_3 = 'Emergency Room visit with tachycardia and violent vertigo.'

ans1_3 = check_concept_3(concept1_3, sentence1_3)
print(ans1_3)   # should print True

concept2_3 = 'Difficulty crying'
sentence2_3 = "Now it's become very hard to cry."

True


In [23]:
concept1_3 = 'Severe vertigo,Severe vertigo'
sentence1_3 = 'Emergency Room visit with tachycardia and violent vertigo.'

assert(check_concept_3(concept1_3, sentence1_3))

In [24]:
concept2_3 = 'Difficulty crying'
sentence2_3 = "Now it's become very hard to cry."

assert(check_concept_3(concept2_3, sentence2_3))

In [25]:
# Test more level 3 examples

In [26]:
# Limit overgeneralization
# We don't want to return True when the entities are not present
# You will not be penalized for identifying level 4 entities here!

## Level 4 - All different words
- These are similar to level 3.
- The main difference will be that none of the words will match exactly.

### Preprocessing 4
- The stub is now left open for you

In [27]:
from string import punctuation as punct
import nltk
from nltk.stem.porter import PorterStemmer
porter_stemmer = PorterStemmer()
# usage:
# porter_stemmer.stem(word)

def text_preprocessing_4_concept(text, stemm=True):
    """Read in text and return a list of tokens
       stemmed, punctuation free, and stop word free"""
    toks = [w.lower() for w in 
                text.translate(text.maketrans(punct, ' '*len(punct))).split()]
    stopwords = nltk.corpus.stopwords.words('english')
    no_stopword = [tok for tok in toks if tok.strip() not in stopwords]
    
    #  KEEP THINKING ABOUT PREPROCESSING  #
    #  DEPEDING ON YOUR IMPLEMENTATION, YOU MAY BE ABLE TO REUSE LEVEL 3  #

    final = no_stopword
    return final 
    
def text_preprocessing_4_sentence(text, stemm=True):
    """Read in text and return a list of tokens
       stemmed, punctuation free, and stop word free"""
    toks = [w.lower() for w in 
                text.translate(text.maketrans(punct, ' '*len(punct))).split()]
    
    stopwords = nltk.corpus.stopwords.words('english')
    no_stopword = [tok for tok in toks if tok.strip() not in stopwords]
    
    #  KEEP THINKING ABOUT PREPROCESSING. #
    #  DEPEDING ON YOUR IMPLEMENTATION, YOU MAY BE ABLE TO REUSE LEVEL 3  #

    final = no_stopword
    return final    

In [28]:

def check_concept_4(concept, sentence):
    import re

    synonyms = [
        {'aggressive', 'aggression', 'hostile', 'hostility',
         'violent', 'belligerent'},
        {'behavior', 'behaviour', 'conduct', 'actions', 'demeanor'},
        {'irregular', 'abnormal', 'erratic', 'unpredictable'},
        {'period', 'periods', 'menstruation', 'menstrual',
         'menses', 'cycle', 'cycles'},
        {'restricted', 'limited', 'impaired', 'impossible',
         'unable', 'inability'},
        {'work', 'job', 'employment'},
        {'performance', 'perform', 'function', 'functioning'},
        {'influenza', 'flu'},
        {'like', 'similar'},
        {'symptoms', 'symptom', 'signs'},
        {'severe', 'intense', 'extreme'},
        {'vertigo', 'dizziness', 'dizzy'}
    ]

    aliases = {}
    for group in synonyms:
        canonical = sorted(group)[0]
        for word in group:
            aliases[word] = canonical

    def preprocess(text):
        text = text.lower()

        text = re.sub(
            r'\b(?:impossible|unable) to do (?:my|the) job\b',
            'restricted work performance',
            text
        )

        text = re.sub(
            r'\bflu[- ]like\b',
            'influenza like symptoms',
            text
        )

        words = re.findall(r'[a-z]+', text)

        return [aliases.get(w, w) for w in words]

    concept_words = preprocess(concept)
    sentence_words = preprocess(sentence)

    return all(
        word in sentence_words
        for word in concept_words
    )


In [29]:
# Level 4 examples
concept1_4 = 'Aggressive behavior'
sentence1_4 = 'They told me that my conduct was hostile'

ans1_4 = check_concept_4(concept1_4, sentence1_4)
print(ans1_4)   # should print True

concept2_4 = 'Irregular periods'
sentence2_4 = 'I\'ve been dealing with abnormal menstruation cycles.'

True


In [30]:
# This cell tests your implementation on these provided examples
concept1_4 = 'Aggressive behavior'
sentence1_4 = 'They told me that my conduct was hostile'
assert(check_concept_4(concept1_4, sentence1_4))

In [31]:
# Test more examples